# Nested feature selection — XGBoost

Leakage-free feature selection for **xgboost** on the 3-class ground-type problem, using the SAME
nested forward-selection harness as the k-NN / SVM notebooks so all five are directly comparable.
Forward selection runs inside each outer leave-user-out fold (features chosen on an inner grouped CV
of that fold's training users only), so the reported macro-F1 has no selection bias.

XGBoost is scale-INVARIANT and robust to redundant features, so — unlike k-NN/SVM — it won't be hurt by carrying extra features. Nested SFS here gives the leakage-free accuracy and the features that actually drive cross-user generalization, to compare against your manual 36-set.

*Honest limit:* 3-class CV rests on 3 users (loose caps it) -> large fold variance; treat gaps under
~3 macro-F1 points as noise. Classifier is UNTUNED (matches the other notebooks' default settings).

In [1]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# ================= CONFIG =================
DATA_DIR='..'
MAX_K=17                       # SFS cap (selection plateaus early)
SFS_TOL=0.0005
SUBSAMPLE_PER_CLASS=None        # XGBoost is cheap; use all data
SFS_JOBS=43                     # thermal-safe parallelism (half of 8 cores)
CLASS_WEIGHT_STRENGTH=0.5
RANDOM_STATE=42

DROP={'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse','heel_mean_ratio',
      'fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'}|{f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META={'foot','step_id','surface','source_user','source_recording'}

dfs=[]
for fn,lab in [('grass_dry.csv','grass'),('loose_dry.csv','loose'),('paved_dry.csv','paved')]:
    d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
df=pd.concat(dfs,ignore_index=True)
FEATS=[c for c in df.columns if c not in META and c not in DROP]
classes=sorted(df.surface.unique()); y=df.surface.values; grp=df.source_user.values
_users_per_class=df.groupby('surface')['source_user'].nunique()
N_SPLITS=int(_users_per_class.min())
_le=LabelEncoder().fit(classes)
print(f'{len(df)} steps | {len(FEATS)} features | users/class {_users_per_class.to_dict()} -> N_SPLITS={N_SPLITS}')
print(f'classifier: xgboost | SUBSAMPLE_PER_CLASS={SUBSAMPLE_PER_CLASS} | SFS_JOBS={SFS_JOBS}')

16971 steps | 49 features | users/class {'grass': 4, 'loose': 4, 'paved': 4} -> N_SPLITS=4
classifier: xgboost | SUBSAMPLE_PER_CLASS=None | SFS_JOBS=43


In [2]:
def _sw(ytr):
    cs,cnt=np.unique(ytr,return_counts=True); n,k=len(ytr),len(cs)
    w={c:(n/(k*ct))**CLASS_WEIGHT_STRENGTH for c,ct in zip(cs,cnt)}
    return np.array([w[v] for v in ytr])

def make_clf():
    return XGBClassifier(max_depth=4, n_estimators=250, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.8, eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0)

def _subsample(idx, yv, per_class, seed=RANDOM_STATE):
    if not per_class: return idx
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(yv[idx]):
        ci=idx[yv[idx]==c]; keep+=list(rng.choice(ci,min(len(ci),per_class),replace=False))
    return np.array(sorted(keep))

Xall=df[FEATS].fillna(0).values

def fit_score(cols, tr, te, yv):
    tr=_subsample(tr, yv, SUBSAMPLE_PER_CLASS)
    m=make_clf(); m.fit(Xall[np.ix_(tr,cols)], _le.transform(yv[tr]), sample_weight=_sw(yv[tr]))
    pred=_le.inverse_transform(m.predict(Xall[np.ix_(te,cols)]))
    return f1_score(yv[te], pred, labels=classes, average='macro')

def inner_cv_score(cols, tr, yv, gv):
    inner=StratifiedGroupKFold(n_splits=max(2,min(N_SPLITS-1,len(np.unique(gv)))),shuffle=True,random_state=0)
    sc=[]
    for itr,ite in inner.split(Xall[tr][:,cols], yv[tr], gv):
        if len(np.unique(yv[tr][itr]))<len(classes): continue
        sc.append(fit_score(cols, tr[itr], tr[ite], yv))
    return np.mean(sc) if sc else -1

In [3]:
# ===== nested forward selection with LIVE progress =====
splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
_folds=[(tr,te) for tr,te in splitter.split(Xall,y,grp) if len(np.unique(y[tr]))==len(classes)]
print(f'nested forward selection for xgboost: {len(_folds)} outer folds, MAX_K={MAX_K}')
outer_mf1=[]; sel_counts={}; per_fold_sets=[]
for _fi,(tr,te) in enumerate(_folds,1):
    t0=time.time(); held=np.unique(grp[te])
    print(f'  fold {_fi}/{len(_folds)} (hold out {held}) — forward-selecting...', flush=True)
    yv,gv=y, grp[tr]
    chosen=[]; remaining=list(range(len(FEATS))); best=-1; step=0
    while remaining and len(chosen)<MAX_K:
        scores=Parallel(n_jobs=SFS_JOBS,prefer='processes')(
            delayed(inner_cv_score)(chosen+[c], tr, yv, gv) for c in remaining)
        s_best,c_best=max(zip(scores,remaining))
        if s_best-best<SFS_TOL: break
        chosen.append(c_best); best=s_best; remaining.remove(c_best); step+=1
        print(f'      + {FEATS[c_best]}  (inner MacroF1 {best*100:.1f}%, {step} feats, {time.time()-t0:.0f}s)', flush=True)
    pf=fit_score(chosen, tr, te, y)
    outer_mf1.append(pf); per_fold_sets.append([FEATS[c] for c in chosen])
    for c in chosen: sel_counts[FEATS[c]]=sel_counts.get(FEATS[c],0)+1
    print(f'    -> fold done: {len(chosen)} feats, held-out MacroF1 {pf*100:.1f}%  ({time.time()-t0:.0f}s)\n', flush=True)
outer_mf1=np.array(outer_mf1)
print(f'=== xgboost: nested (leakage-free) grouped MacroF1 {outer_mf1.mean()*100:.1f}% '
      f'(sd {outer_mf1.std()*100:.1f}) | per-fold {np.round(outer_mf1*100,1)} ===')

nested forward selection for xgboost: 4 outer folds, MAX_K=17
  fold 1/4 (hold out ['Sam']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 45.4%, 1 feats, 10s)
      + accel_spec_entropy  (inner MacroF1 51.1%, 2 feats, 15s)
      + accel_jerk_ratio  (inner MacroF1 52.4%, 3 feats, 21s)
      + orient_rest_b_roll3_std  (inner MacroF1 53.4%, 4 feats, 26s)
      + gu_press_jitter  (inner MacroF1 54.9%, 5 feats, 32s)
      + accel_spec_centroid  (inner MacroF1 55.4%, 6 feats, 38s)
      + gu_active_sensors  (inner MacroF1 55.7%, 7 feats, 44s)
    -> fold done: 7 feats, held-out MacroF1 49.7%  (50s)

  fold 2/4 (hold out ['Kristian']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 47.5%, 1 feats, 5s)
      + accel_hf_frac  (inner MacroF1 51.3%, 2 feats, 11s)
      + gu_fore_heel_toff  (inner MacroF1 51.9%, 3 feats, 17s)
      + gyro_x_std  (inner MacroF1 52.8%, 4 feats, 23s)
      + accel_jerk_ratio  (inner MacroF1 53.7%, 5 feats, 29s)
      + 

In [4]:
# ===== selection frequency + comparison to your manual 36-set =====
freq=sorted(sel_counts.items(), key=lambda kv:-kv[1])
print(f'Feature selection frequency for xgboost (across {len(_folds)} folds):')
for f,c in freq: print(f'  {c}/{len(_folds)}  {f}')
maj=[f for f,c in freq if c> len(_folds)//2]
print(f'\nStable set (selected in > half of folds): {maj}')

Feature selection frequency for xgboost (across 4 folds):
  4/4  orient_rest_a_roll3_std
  4/4  accel_spec_entropy
  2/4  accel_jerk_ratio
  2/4  gu_press_jitter
  2/4  stance_duration_roll3_std
  2/4  contact_time_sec
  2/4  time_to_peak_frac
  2/4  fall_slope
  1/4  orient_rest_b_roll3_std
  1/4  accel_spec_centroid
  1/4  gu_active_sensors
  1/4  accel_hf_frac
  1/4  gu_fore_heel_toff
  1/4  gyro_x_std
  1/4  slope_asym
  1/4  orient_range_b
  1/4  loading_rate_norm
  1/4  loading_rate_roll3_std

Stable set (selected in > half of folds): ['orient_rest_a_roll3_std', 'accel_spec_entropy']


## How to read this
- **Nested MacroF1** is xgboost's honest, leakage-free cross-user accuracy — report this as the number.
- **Stable set** = features selected in a majority of folds; the trustworthy takeaway (single-fold picks vary with only 3 users).

